In [ ]:
import matplotlib.pyplot as plt
import cv2
import json
import pandas as pd
import numpy as np
import glob

# Clase DataHandPDAnalysis

In [ ]:
from ast import Dict

class DataPreProcess:

  def __init__(self, test_type, t_group):
    self.test_type = test_type
    self.t_group = t_group
    self.folder = f'{test_type}{t_group}'
    self.path = f"/content/drive/MyDrive/{self.folder}_out/"
    self.path_transform = f"{self.folder}_out_resize"
    self.struct = []

  def load_data(self, path):
    data = cv2.imread(path)
    return data

  def save_data(self, data, path_in):
    #print(path_in)
    path = path_in.replace(f"{self.folder}_out", self.path_transform)
    print(path)
    cv2.imwrite(path, data)

  def run_study(self):
    for i in glob.glob(self.path+"*_TH.jpg"):
      data_img = self.load_data(i)
      name_img = i.split("/")[-1]
      data_sh = data_img.shape
      self.struct.append(dict(user_proof=name_img,
                              user = name_img.split("-")[0],
                              w=data_sh[0],
                              h=data_sh[1],
                              t_group=self.t_group))
    #
    df = self.build_dataset(self.struct)
    #
    return df

  def apply_transform(self, data_img, new_size):
    #print(data_img)
    return cv2.resize(data_img, (new_size[0], new_size[1]))

  def run_transform(self, new_size):
    for i in glob.glob(self.path+"*_TH.jpg"):
      data = self.load_data(i)
      new_img = self.apply_transform(data, new_size)
      self.save_data(new_img, i)

  def build_dataset(self, dict_data):
    df = pd.DataFrame(dict_data)
    return df

# Ejecución
Se estudia el tamaño de las distintas imagenes disponibles.

In [ ]:
list_exe = [("Spiral","Control"), ("Spiral","Patients"),
            ("Meander","Control"), ("Meander","Patients")]
result_dfs = []
for i in list_exe:
  pread = DataPreProcess(i[0], i[1])
  df = pread.run_study()
  result_dfs.append(df)

Se genera dataframe para estudiar las caracteristicas de cada tipo de prueba.

In [ ]:
df_data_spiral = pd.merge(result_dfs[0], result_dfs[1], how='outer')
df_data_meander = pd.merge(result_dfs[2], result_dfs[3], how='outer')

Estudio de la distribución de las etiquetas de los datos

In [ ]:
df_user_group_spiral = df_data_spiral[['t_group','user']].drop_duplicates()
df_user_group_meander = df_data_meander[['t_group','user']].drop_duplicates()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 10))
df_data_spiral['t_group'].value_counts().plot(kind='bar',
                                               ax=ax[0],
                                               title='Distribution of t_group (Spiral Data)')
df_user_group_spiral.groupby(['t_group']).count().plot(kind='bar',
                                                ax=ax[1],
                                                title='Distribution of t_group (Unique User-Group Spiral Data)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 10))
df_data_meander['t_group'].value_counts().plot(kind='bar',
                                               ax=ax[0],
                                               title='Distribution of t_group (Meander Data)')
df_user_group_meander.groupby(['t_group']).count().plot(kind='bar',
                                                ax=ax[1],
                                                title='Distribution of t_group (Unique User-Group Meander Data)')
plt.tight_layout()
plt.show()

Estudio de las dimensiones de las imagenes

In [ ]:
#.max()#.sort_values(ascending=False) --> 513,532 |||  520,536  650, 644]
print(f"{df_data_spiral['w'].min()},{df_data_spiral['h'].min()}")

In [ ]:
print(f"{df_data_spiral['w'].min()},{df_data_spiral['h'].max()}")

644,714


In [ ]:
df_data_spiral[['w','h']].value_counts()

In [ ]:
print(f"{df2['w'].min()},{df2['h'].min()}") #.max()#.sort_values(ascending=False) --> 513,532 |||  520,536

Partiendo del análisis anterior de aplica la redimención de las imagenes

In [ ]:
list_exe = [("Spiral","Control"), ("Spiral","Patients"),
            ("Meander","Control"), ("Meander","Patients")]
for i in list_exe:
  pread = DataPreProcess(i[0], i[1])
  pread.run_transform((650,644))